# Data Science for Business - Gradient Descent for Linear Regression

*Session 3 · ISLP 3.1.1 (eq. 3.4, Table 3.1, Fig. 3.2) · 10.7 (eqs. 10.25–10.27, Fig. 10.17)*

We estimate the coefficients $\beta_0, \beta_1$ of a simple linear regression iteratively with gradient descent, instead of with the closed-form least squares solution (ISLP eq. 3.4). The same algorithm is used later to fit neural networks (ISLP 10.7).

**Warm-up:** `gradient_descent.xlsx` (in this session's folder) performs gradient descent by hand for the one-dimensional function $f(x)=(x-3)^2$, starting at $x=5.2$ with learning rate 0.1. Work through it first.

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import statsmodels.formula.api as smf

In [ ]:
rng = np.random.default_rng(42)
plt.style.use('fivethirtyeight')

## Problem description

How are sales related to TV advertising? This is the *Advertising* data from ISLP Ch. 2 and 3: `Sales` (in thousands of units) and the advertising budget for `TV` (in thousands of dollars) in 200 markets. We fit the simple linear regression $\text{Sales} \approx \beta_0 + \beta_1 \times \text{TV}$, as in ISLP Table 3.1.

## Load data

Advertising data, as in ISLP Ch. 3.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_03/advertising.csv')

In [ ]:
data.head()

## Prepare data

Extract the response $y$ (`Sales`) and the predictor $x$ (`TV`) as NumPy arrays.

In [ ]:
y = data['Sales'].to_numpy()
x = data['TV'].to_numpy()

## Initialize parameters

Random start values for $\beta_0$ (intercept) and $\beta_1$ (slope). The learning rate $\rho$ sets the step size: too large and gradient descent diverges, too small and it is slow. Because our loss is a *sum* over all $n=200$ observations (not an average), a suitable $\rho$ also depends on $n$ and on the scale of $x$ (TV budgets go up to ~300).

In [ ]:
beta0, beta1 = rng.uniform(0, 1, size=2)

rho = 1e-7          # learning rate ρ in ISLP (10.27)
n_iter = 1_000_000  # number of gradient descent steps

print("Initial values of beta0 and beta1:", beta0, beta1)

## Run gradient descent

We minimise the residual sum of squares

$$R(\beta)=\sum_{i=1}^n(y_i-\beta_0-\beta_1x_i)^2$$

(ISLP 3.1.1, eq. 10.25 up to a factor ½). Its partial derivatives are

$$\frac{\partial R}{\partial \beta_0} = 2\sum_i(\beta_0+\beta_1x_i-y_i), \qquad \frac{\partial R}{\partial \beta_1} = 2\sum_i x_i(\beta_0+\beta_1x_i-y_i).$$

In each step we move against the gradient: $\beta \leftarrow \beta-\rho\,\nabla R(\beta)$ (ISLP 10.27). ISLP iterates until the objective fails to decrease; for simplicity, we run a fixed number of steps. Every 10,000 steps we store the RSS to plot the loss curve later. This takes a few seconds.

In [ ]:
rss_history = []

for i in range(1, n_iter + 1):

    # Residuals of the current model (computed once per step)
    residuals = beta0 + beta1 * x - y

    # Partial derivatives of the RSS with respect to beta0 and beta1
    d_beta0 = 2 * residuals.sum()
    d_beta1 = 2 * (x * residuals).sum()

    # Take a step in the opposite direction of the gradient
    beta0, beta1 = beta0 - rho * d_beta0, beta1 - rho * d_beta1

    # Store the RSS and print the estimates every 10,000 steps
    if i % 10_000 == 0:
        rss_history.append((residuals**2).sum())
    if i % 100_000 == 0:
        print("Iteration:", i, "| Intercept:", round(beta0, 6), "| Slope:", round(beta1, 6))

Final estimates.

In [ ]:
print("Final estimated intercept:", round(beta0, 4), "| Final estimated slope:", round(beta1, 4))

### Loss curve

The RSS falls quickly at first and then flattens out as we approach the minimum. (ISLP Fig. 3.2 shows the RSS surface over $\beta_0, \beta_1$; gradient descent walks downhill on it.)

In [ ]:
plt.plot(np.arange(1, len(rss_history) + 1) * 10_000, rss_history)
plt.xlabel('Iteration')
plt.ylabel('RSS')
plt.show()

## Compare with OLS

For comparison, the closed-form least squares solution (ISLP eq. 3.4) computed by statsmodels. Both should match ISLP Table 3.1 (intercept 7.0325, slope 0.0475) up to rounding.

In [ ]:
ols = smf.ols('Sales ~ TV', data=data).fit()
print(ols.summary())

## Your turn!

**Your task:** Re-run the *Initialize parameters* and *Run gradient descent* cells with $\rho$ = 1e-6 and $\rho$ = 1e-8 (you may want to lower `n_iter` first, e.g., to 100,000). What happens to the estimates and the loss curve?

In [ ]:
# YOUR CODE HERE